In [ ]:
%pip install unidecode
%pip install nltk
%pip install networkx
%pip install scikit-learn

In [ ]:
import nltk
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

import json

import keras
import keras_hub
import tensorflow as tf

import networkx as nx
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from unidecode import unidecode

In [ ]:
CORPUS_PATH = "corpus_graph_eu.json"
ONTOLOGY_PATH = "ontology_graph_eu.json"

PRETRAINED_BERT = "/kaggle/input/datasets/mattiaingrassia/bert-trained-on-wikivoyage/K-COG-BERT.keras"

lemmatizer = WordNetLemmatizer()

In [ ]:
def load_graph(graph_path, include_located_in=False):
    with open(graph_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    G = nx.DiGraph()
    for node in data['nodes']:
        G.add_node(node)
    for edge in data['edges']:
        if edge['relation'] == "type":
            G.add_edge(edge['from'], edge['to'], relation=edge['relation'])
        elif include_located_in and edge['relation'] == "located_in":
            G.add_edge(edge['from'], edge['to'], relation=edge['relation'])
    return G


# Lowercase + replace _ with space + lemmatize
def normalize_label(label):
    normalized_label = unidecode(label)
    normalized_label = normalized_label.replace('_', ' ').lower()
    words = word_tokenize(normalized_label)
    return ' '.join(lemmatizer.lemmatize(word) for word in words)

def preprocess_graph(graph):
    names_mapping = {node: normalize_label(node) for node in graph}
    return nx.relabel_nodes(graph, names_mapping)

In [ ]:
POINTS_OF_INTEREST = [
    "tourist_attraction", "archaeological_site", "monument", "museum",
    "palace", "monastery", "cathedral", "square", "castle",
    "tower", "park", "lake", "theatre", "stadium", "gallery",
    "river", "beach", "waterfall", "mountain", "market", "church",
    "synagogue", "mosque"
]

NORMALIZED_POINTS_OF_INTEREST = {normalize_label(t) for t in POINTS_OF_INTEREST}

In [ ]:
def find_shared_concepts(corpus_graph, reference_ontology):
    corpus_nodes = set(corpus_graph)
    ontology_nodes = set(reference_ontology)
    shared_concepts = corpus_nodes.intersection(ontology_nodes)
    shared_concepts = shared_concepts - NORMALIZED_POINTS_OF_INTEREST
    return shared_concepts

def get_matching_graph(reference_ontology, shared_concepts):
    nodes_to_include = set(shared_concepts)
    for node in shared_concepts:
        if node in reference_ontology:
            ancestors = nx.ancestors(reference_ontology, node)
            children = reference_ontology.successors(node)
            nodes_to_include.update(ancestors)
            nodes_to_include.add(node)
            nodes_to_include.update(children)


    edges_to_include = []
    for u, v, data in reference_ontology.edges(data=True):
        if u in nodes_to_include and v in nodes_to_include:
            edges_to_include.append((u, v, data))


    G = nx.DiGraph()
    G.add_nodes_from(nodes_to_include)
    G.add_edges_from(edges_to_include)
    return G

In [ ]:
def save_final_graph_exported(reference_ontology, corpus_ontology, shared_concepts):
    shared_concepts_extra = set(shared_concepts).union(NORMALIZED_POINTS_OF_INTEREST)

    # Nodes
    reference_nodes = set(reference_ontology.nodes)
    corpus_nodes    = set(corpus_ontology.nodes)

    node_status = dict.fromkeys(shared_concepts_extra, "SHARED")
    for n in reference_nodes - shared_concepts_extra:
        node_status[n] = "REFERENCE_ONLY"
    for n in corpus_nodes - shared_concepts_extra - reference_nodes:
        node_status[n] = "CORPUS_ONLY"

    # Edges
    corpus_edge_set = {
        (s, t, d["relation"]) for s, t, d in corpus_ontology.edges(data=True)
    }

    edge_status = {}
    for s, t, d in reference_ontology.edges(data=True):
        triple = (s, t, d["relation"])
        edge_status[triple] = "SHARED" if triple in corpus_edge_set else "REFERENCE_ONLY"

    for s, t, d in corpus_ontology.edges(data=True):
        triple = (s, t, d["relation"])
        if triple not in edge_status:
            edge_status[triple] = "CORPUS_ONLY"

    # Serialize
    result_json = {
        "nodes": [{"name": n, "status": s} for n, s in node_status.items()],
        "edges": [{"from": s, "to": t, "relation": r, "status": st}
                  for (s, t, r), st in edge_status.items()],
    }

    with open("final_graph_matching.json", "w", encoding="utf-8") as f:
        json.dump(result_json, f, indent=4, ensure_ascii=False)

In [ ]:
# Load corpus
corpus_graph = load_graph(CORPUS_PATH)
reference_ontology = load_graph(ONTOLOGY_PATH)

corpus_graph = preprocess_graph(corpus_graph)
reference_ontology = preprocess_graph(reference_ontology)

print(f'Corpus graph: {len(corpus_graph):,} nodes, {corpus_graph.number_of_edges():,} edges')
print(f'Reference ontology : {len(reference_ontology):,} nodes, {reference_ontology.number_of_edges():,} edges')

shared_concepts = find_shared_concepts(corpus_graph, reference_ontology)

print(f'Shared concepts : {len(shared_concepts):,} node')

corpus_with_located = preprocess_graph(load_graph(CORPUS_PATH, include_located_in=True))
reference_with_located = preprocess_graph(load_graph(ONTOLOGY_PATH, include_located_in=True))
save_final_graph_exported(reference_with_located, corpus_with_located, shared_concepts)

sub_ontology = get_matching_graph(reference_ontology, shared_concepts)
print(f'Sub-ontology : {len(sub_ontology):,} nodes, {sub_ontology.number_of_edges():,} edges')

In [ ]:
# Model 1 - BERT base untouched

preprocessor = keras_hub.models.BertTextClassifierPreprocessor.from_preset(
    "bert_base_en_uncased",
    sequence_length=512
)

bert_model = keras_hub.models.Backbone.from_preset(
    "bert_base_en_uncased"
)

last_layer_output = bert_model.get_layer("transformer_layer_11").output
bert_model = keras.Model(
        inputs=bert_model.inputs,
        outputs=[last_layer_output]
    )


# Modello 2: Bert base pre-trained on wikivoyage
bert_model_pt = keras.models.load_model(PRETRAINED_BERT)

In [ ]:
print("O: Number of concepts in the candidate ontology")
O = len(reference_ontology)

print("D: Number of concepts extracted from the domain (corpus) after pruning")
D = len(corpus_graph)

print("S: Number of shared concepts found between the candidate ontology and the domain concepts")
S = len(shared_concepts)

print("H: Number of concepts in the subset of the candidate ontology constructed by taking the shared concepts and expanding them using the ontology hierarchical relations")
H = len(sub_ontology)


print("\n\n")
print(f'D={D}  O={O}  S={S}  H={H}')
print(f'Domain Coverage        S/D = {S/D:.4f}')
print(f'Ontology Relevance     S/O = {S/O:.4f}')
print(f'Sub-Ontology Relevance S/H = {S/H:.4f}')

result_output = {}
result_output['D'] = D
result_output['O'] = O
result_output['S'] = S
result_output['H'] = H
result_output['Domain Coverage'] = round(S/D, 4)
result_output['Ontology Relevance'] = round(S/O, 4)
result_output['Sub-Ontology Relevance'] = round(S/H, 4)

### Def 2.1 (Ontology)

Let $C$ be a set of concepts, $R$ a set of relationships, and $A$ a set of associations such that:

$$
A \subseteq \{ r(x, y) \mid \forall r \in R, \forall x \in C, \forall y \in C \}
$$

An ontology $O$ is therefore defined as the triple

$$
O = \langle C, R, A \rangle
$$

### Def 2.2 (Concept Family)

Let $O = \langle C, R, A \rangle$ be an ontology and let $r$ (IS-A) $\in R$ be one of its relationships.

Then, $CF$ is a *concept family* if and only if:

$$
CF = \langle C_p, C_s \rangle
$$

Where:
- $C_p \in C$
- $C_s \subseteq C$
- $ \forall c \in C_s \Rightarrow r $ (IS-A) $(c, C_p) $


In other words, $C_p$ is the parent node and $C_s$ denotes the child nodes.
Each child node must be connected to the parent node through an IS-A relationship



In [ ]:
def get_concept_families(relationships, sub_ontology):
    concept_families = {}
    for relationship in relationships:
        concept_families[relationship] = {}
        for v, w, data in sub_ontology.edges.data("relation"):
            if data != relationship:
                continue
            # W parent, V child
            if w not in concept_families[relationship]:
                concept_families[relationship][w] = [v]
            else:
                concept_families[relationship][w].append(v)
    return concept_families


### CSS

Child Similarity Score
- A metric used to assess the accuracy as it evaluates the correctness of the Concept Family (CF) as constructed.
- Defined as the average cosine similarity among all pairs of sibling concepts within the same Concept Family.
- $M$ denotes the number of child concepts in the Concept Family.

$$
CSS(CF) = \frac{2}{M(M-1)} \sum_{i=1}^{M-1} \sum_{j=i+1}^M similarity(C_i, C_j)
$$

In [ ]:
def child_similarity_score(children, cached_encodings):
    M = len(children)
    node_mean_similarities = []
    for i in range(M - 1):
        current_node_similarity = 0
        for j in range (i + 1, M):
            emb_i = cached_encodings[children[i]]
            emb_j = cached_encodings[children[j]]
            current_node_similarity += cosine_similarity(emb_i, emb_j)[0][0]
        node_mean_similarities.append(current_node_similarity / (M - 1))
    css = 2.0 * sum(node_mean_similarities) / M
    return css


### PSS

Parent Similarity Score
- Measures consistency by evaluating the extent to which the same relationship (is-A) within a Concept Family agrees with the parent concept.
- Average cosine similarity between the parent concept and its direct child concepts.
- $M$ = number of child concepts in a Concept Family (CF).
- $C_p$ = parent concept.

$$
PSS(CF) = \frac{1}{M} \sum_{i=1}^{M} \text{similarity}(C_i, C_p)
$$

In [ ]:
def parent_similarity_score(parent, children, cached_encodings):
    M = len(children)
    total_similarity = 0
    emb_parent = cached_encodings[parent]
    for i in range(M):
        emb_i = cached_encodings[children[i]]
        current_similarity = cosine_similarity(emb_i, emb_parent)[0][0]
        total_similarity += current_similarity
    pss = total_similarity / M
    return pss


### PDA

Parent Difference Agreement
- Measures consistency by evaluating the extent to which the same relationship (is-A) within a Concept Family agrees with the parent concept.
- Uses the standard deviation of the similarity between the parent concept and its direct child concepts.
- (Degree of agreement among siblings with respect to the parent)
- $M$ = number of child concepts in a Concept Family (CF).
- $C_p$ = parent concept.

$$
PDA(CF) =  1 - \sqrt{\frac{1}{M-1} \sum_{i=1}^{M} \left[ \text{similarity}(C_i, C_p) - PSS(CF) \right]^2}
$$

In [ ]:
def parent_difference_agreement(parent, children, cached_encodings, pss_value):
    M = len(children)
    if M == 1:
        return 1
    emb_parent = cached_encodings[parent]
    total_similarity = 0
    for i in range(M):
        emb_i =  cached_encodings[children[i]]
        current_similarity = cosine_similarity(emb_i, emb_parent)[0][0]
        sum_term = (current_similarity - pss_value) ** 2
        total_similarity += sum_term
    pda = 1 - (( 1 / (M - 1)) * total_similarity ) ** 0.5
    return pda


In [ ]:
relationships = list(set(nx.get_edge_attributes(sub_ontology, 'relation').values()))

concept_families = get_concept_families(relationships, sub_ontology)

all_nodes = set()
for relationship, families in concept_families.items():
    for parent, children in families.items():
        all_nodes.add(parent)
        all_nodes.update(children)

all_nodes = list(all_nodes)

In [ ]:
# Tokenize all nodes only one time
preprocessed = preprocessor(all_nodes)
BATCH_SIZE = 32
dataset = tf.data.Dataset.from_tensor_slices(preprocessed).batch(BATCH_SIZE)

def generate_embeddings(bert_model, dataset, preprocessed):
    print(f"Encoding {len(all_nodes)} nodes")
    # Mean pooling with mask - exclude CLS, SEP and padding
    sequence_output = bert_model.predict(dataset)
    padding_mask    = preprocessed["padding_mask"].numpy()
    content_mask = padding_mask.copy()
    # Remove CLS
    content_mask[:, 0] = 0
    # Remove SEP
    sep_positions = padding_mask.sum(axis=1).astype(int) - 1
    for i, sep_pos in enumerate(sep_positions):
        content_mask[i, sep_pos] = 0

    mask_expanded  = content_mask[:, :, np.newaxis]
    sum_embeddings = np.sum(sequence_output * mask_expanded, axis=1)
    sum_mask = content_mask.sum(axis=1, keepdims=True).clip(min=1e-9)
    embeddings = sum_embeddings / sum_mask
    return embeddings

In [ ]:
embeddings = generate_embeddings(bert_model, dataset, preprocessed)
embeddings_pt = generate_embeddings(bert_model_pt, dataset, preprocessed)

cached_embeddings = {name: embeddings[i].reshape(1,-1) for i, name in enumerate(all_nodes)}
cached_embeddings_pt = {name: embeddings_pt[i].reshape(1,-1) for i, name in enumerate(all_nodes)}
print("Embeddings encoded correctly")

In [ ]:
def compute_and_store_scores(concept_families, cached_embeddings, model_key):
    total = sum(len(families) for families in concept_families.values())
    css_scores, pss_scores, pda_scores = [], [], []
    result_output[model_key] = {}
    done = 0
    for relationship, families in concept_families.items():
        for parent, children in families.items():
            css = child_similarity_score(children, cached_embeddings)
            pss = parent_similarity_score(parent, children, cached_embeddings)
            pda = parent_difference_agreement(parent, children, cached_embeddings, pss)

            css_scores.append(css)
            pss_scores.append(pss)
            pda_scores.append(pda)

            result_output[model_key][parent] = {
                "CSS": float(css),
                "PSS": float(pss),
                "PDA": float(pda),
            }

            done += 1
            print(f"Progress: {done}/{total} ({done/total:.1%})")

    mean_css = float(np.mean(css_scores))
    mean_pss = float(np.mean(pss_scores))
    mean_pda = float(np.mean(pda_scores))
    print(f'{model_key}')
    print(f'Families scored: {len(css_scores)}')
    print(f'Mean CSS = {mean_css:.4f}  (Cohesion of siblings)')
    print(f'Mean PSS = {mean_pss:.4f}  (Coerence parent-child)')
    print(f'Mean PDA = {mean_pda:.4f}  (CF Consistency)')

    result_output[model_key]["mean_CSS"] = mean_css
    result_output[model_key]["mean_PSS"] = mean_pss
    result_output[model_key]["mean_PDA"] = mean_pda


In [ ]:
print("Calculating CSS, PSS, PDA scores")
compute_and_store_scores(concept_families=concept_families, cached_embeddings=cached_embeddings, model_key="BERT")
compute_and_store_scores(concept_families=concept_families, cached_embeddings=cached_embeddings_pt, model_key="KCOG-BERT")

In [ ]:
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(result_output, f, indent=4, ensure_ascii=False)